In [ ]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

In [ ]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, re, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- THEME & CONFIG ---
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f9fcfc", "bg_sidebar": "#e3f6f5", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#ffd803", "accent_hover": "#e6c300", "accent_text": "#272343",
    "border": "#272343", "border_light": "#bae8e8", "success": "#34d399", "danger": "#f87171"
}

# --- SECRETS ---
JWT_SECRET = os.getenv("JWT_SECRET", "super-secret-infosys-key-2026")
SENDER_EMAIL = os.getenv("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
OTP_EXPIRY_MINUTES = 5
ADMIN_USER = "admin"
ADMIN_PASS = "Admin@123" # Hardcoded per Milestone requirements

# --- CSS STYLING ---
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}
    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{ background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important; box-shadow: 6px 6px 0px {COLORS['border']} !important; }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

# --- DATABASE & UTILS ---
def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")

# --- VALIDATORS ---
def validate_email(email):
    # Enforces: 2+ letters before @, 2+ letters between @ and dot, 2+ letters after final dot
    pattern = r"^[a-zA-Z]{2,}[a-zA-Z0-9._-]*@[a-zA-Z]{2,}[a-zA-Z0-9.-]*\.[a-zA-Z]{2,}$"
    return bool(re.match(pattern, email))

def validate_password(pwd):
    # Enforces: Min 8 chars, 1 uppercase, 1 lowercase, 1 number, 1 special symbol
    pattern = r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&])[A-Za-z\d@$!%*?&]{8,}$"
    return bool(re.match(pattern, pwd))

# --- JWT & OTP LOGIC ---
def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None
def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"
def make_otp_token(email, otp):
    payload = {"sub": email, "otp_hash": hash_txt(otp), "type": "otp", "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def send_otp_email(to_email, otp, app_pass):
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Infosys Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Infosys Portal - Verification Code"
    text_body = f"Your code is: {otp}\nExpires in {OTP_EXPIRY_MINUTES} minutes."
    msg.attach(MIMEText(text_body, 'plain'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(SENDER_EMAIL, app_pass if app_pass else "")
        s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        s.quit()
        return True, "Email sent successfully!"
    except Exception as e: return False, f"SMTP Error: {str(e)}"

# --- SESSION STATE ---
for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None), ("jwt_otp", None)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div><h1 style="font-size:2rem !important;margin:0;">Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]: st.session_state.page = "Login"
    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        # --- LOGIN PAGE ---
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")

            # 🚀 Create the visual separation using Streamlit Tabs
            tab_user, tab_admin = st.tabs(["👤 User Portal", "🛡️ Admin Access"])

            # --- USER LOGIN ROUTE ---
            with tab_user:
                st.markdown("<br>", unsafe_allow_html=True)
                user_input = st.text_input("Username or Email address", key="u_id").strip()
                pwd = st.text_input("Password", type="password", key="u_pwd")
                st.markdown("<br>", unsafe_allow_html=True)

                col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
                if col_l.button("Sign In →", key="btn_user_login", use_container_width=True):
                    if not user_input or not pwd:
                        st.error("⚠️ Username/Email and Password are required.")
                    else:
                        with get_db() as c:
                            r = c.execute("SELECT email, password_hash FROM users WHERE email=? OR username=?", (user_input.lower(), user_input)).fetchone()
                        if r and check_txt(pwd, r[1]):
                            st.session_state.token = make_jwt(r[0]); navigate("Dashboard")
                        else: st.error("❌ Login failed. The credentials provided are incorrect.")

                if col_c.button("Create Account", use_container_width=True): navigate("Signup")
                if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

            # --- ADMIN LOGIN ROUTE ---
            with tab_admin:
                st.markdown("<br>", unsafe_allow_html=True)
                st.info("Restricted Area: Authorized System Administrators Only.")
                admin_id = st.text_input("Admin ID", key="a_id").strip()
                admin_pwd = st.text_input("Admin Password", type="password", key="a_pwd")
                st.markdown("<br>", unsafe_allow_html=True)

                if st.button("Authenticate Admin →", key="btn_admin_login", use_container_width=True):
                    if not admin_id or not admin_pwd:
                        st.error("⚠️ Credentials required.")
                    elif admin_id == ADMIN_USER and admin_pwd == ADMIN_PASS:
                        st.session_state.token = make_jwt(ADMIN_USER); navigate("Dashboard")
                    else:
                        st.error("❌ Invalid Administrator Credentials.")

        # --- SIGNUP PAGE ---
        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Username")
            email = st.text_input("Email address").lower().strip()
            pwd = st.text_input("Password", type="password")
            confirm_pwd = st.text_input("Confirm password", type="password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not all([uname, email, pwd, confirm_pwd, sa]):
                    st.error("⚠️ All fields are mandatory.")
                elif not validate_email(email):
                    st.error("❌ Invalid email format (Requires min. 2 letters before '@', between '@' and '.', and after '.').")
                elif not validate_password(pwd):
                    st.error("❌ Password must be min 8 chars, contain 1 uppercase, 1 lowercase, 1 number, and 1 special symbol.")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.success("✅ Account created! Please log in."); time.sleep(1); navigate("Login")
                    except sqlite3.IntegrityError:
                        st.error("❌ Username or Email already exists. Please choose another.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        # --- FORGOT PASSWORD PAGE ---
        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            if not st.session_state.reset_email:
                user_input = st.text_input("Registered username or email").strip()
                st.markdown("<br>", unsafe_allow_html=True)
                col_sq, col_otp = st.columns(2)

                # Route 1: Security Question
                if col_sq.button("Via Security Question", use_container_width=True):
                    if not user_input: st.error("⚠️ Please enter your username or email.")
                    else:
                        with get_db() as c: r = c.execute("SELECT email, security_question FROM users WHERE email=? OR username=?", (user_input.lower(), user_input)).fetchone()
                        if r:
                            st.session_state.reset_email, st.session_state.sq_p, st.session_state.reset_mode = r[0], r[1], "sq"
                            st.rerun()
                        else: st.error("❌ Account not found.")

                # Route 2: OTP
                if col_otp.button("Via OTP Email", use_container_width=True):
                    if not user_input: st.error("⚠️ Please enter your username or email.")
                    elif not EMAIL_PASSWORD or not SENDER_EMAIL: st.error("❌ Email secrets not configured in Colab.")
                    else:
                        with get_db() as c: r = c.execute("SELECT email FROM users WHERE email=? OR username=?", (user_input.lower(), user_input)).fetchone()
                        if r:
                            target_email = r[0]
                            otp = generate_otp()
                            ok, msg = send_otp_email(target_email, otp, EMAIL_PASSWORD)
                            if ok:
                                st.session_state.reset_email, st.session_state.jwt_otp, st.session_state.reset_mode = target_email, make_otp_token(target_email, otp), "otp"
                                st.success("✅ OTP Sent!"); time.sleep(1); st.rerun()
                            else: st.error(f"❌ {msg}")
                        else: st.error("❌ Account not found.")

            # Active Reset Flows
            else:
                st.info(f"Account: **{st.session_state.reset_email}**")

                # Validate Answer / OTP
                auth_passed = False
                if st.session_state.reset_mode == "sq":
                    ans = st.text_input(f"Question: {st.session_state.sq_p}").lower().strip()
                    with get_db() as c: stored_ans = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()[0]
                    if ans: auth_passed = check_txt(ans, stored_ans)

                elif st.session_state.reset_mode == "otp":
                    otp_input = st.text_input("Enter 6-Digit Verification Code")
                    if len(otp_input) == 6:
                        try:
                            payload = jwt.decode(st.session_state.jwt_otp, JWT_SECRET, algorithms=["HS256"])
                            auth_passed = check_txt(otp_input, payload["otp_hash"])
                        except: st.error("⚠️ OTP expired or invalid.")

                # If authenticated, allow new password entry
                if auth_passed:
                    st.success("✅ Verification successful. Enter a new password.")
                    npw = st.text_input("New password", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    if st.button("Reset Password →", use_container_width=True):
                        if not validate_password(npw): st.error("❌ Password must be min 8 chars, 1 uppercase, 1 lowercase, 1 number, 1 special symbol.")
                        elif npw != confirm_npw: st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                            st.success("✅ Password updated successfully!"); time.sleep(1); st.session_state.reset_email = None; navigate("Login")
                elif (st.session_state.reset_mode == "sq" and ans) or (st.session_state.reset_mode == "otp" and len(otp_input)==6):
                    st.error("❌ Verification failed. Incorrect answer or OTP.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = None; st.session_state.reset_mode = None; navigate("Login")

# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload: st.session_state.token = None; navigate("Login")

    current_user = payload["email"]

    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div><div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if current_user==ADMIN_USER else "User Dashboard"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if current_user == ADMIN_USER else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if current_user==ADMIN_USER else ["house", "graph-up", "file-text", "box-arrow-right"])
        if menu == "Logout": st.session_state.token = None; navigate("Login")

    # --- ADMIN DASHBOARD ---
    if current_user == ADMIN_USER:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']}; border-radius:16px; padding:24px 32px; display:flex; justify-content:space-between; align-items:center; margin-bottom:24px;">
            <div style="color:{COLORS['bg_main']}; font-size:26px; font-weight:700; font-family:'Poppins', sans-serif;">
                <span style="color:{COLORS['accent']};">⚡</span> Admin Control Panel
            </div>
        </div>
        """, unsafe_allow_html=True)

        with get_db() as c: users_list = c.execute("SELECT id, username, email FROM users").fetchall()

        st.markdown("### Registered Users Directory")
        if users_list:
            # Display safely without passwords as required
            import pandas as pd
            df = pd.DataFrame(users_list, columns=["ID", "Username", "Email"])
            st.dataframe(df, use_container_width=True, hide_index=True)
        else:
            # Handles the edge case if no users have signed up yet
            st.info("No registered users found in the system yet.")

    # --- USER DASHBOARD ---
    else:
        with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (current_user,)).fetchone()[0]
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']}; border-radius:16px; padding:24px 32px; display:flex; justify-content:space-between; align-items:center; margin-bottom:24px;">
            <div style="color:{COLORS['bg_main']}; font-size:26px; font-weight:700; font-family:'Poppins', sans-serif;">
                <span style="color:{COLORS['accent']};">⚡</span> Welcome back, {uname}
            </div>
        </div>
        """, unsafe_allow_html=True)

        # Make sure you didn't accidentally delete your metrics cards below this!
        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents", "128"), (c2, "🔍", "Searches", "47"), (c3, "📊", "Efficiency", "98.4%"), (c4, "🛡️", "Security", "Secured")]:
            col.markdown(f"<div class='pn-card' style='text-align:center;'><div style='font-size:28px;'>{icon}</div><div style='font-size:26px;font-weight:700;'>{val}</div><div style='color:{COLORS['text_muted']};font-size:12px;font-weight:600;'>{lbl}</div></div>", unsafe_allow_html=True)

In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Fetch secrets from Colab UI and push them into the OS environment
try:
    os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
    os.environ['EMAIL_ADDRESS'] = userdata.get('EMAIL_ADDRESS')
    os.environ['JWT_SECRET'] = userdata.get('JWT_SECRET')
    NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
except Exception as e:
    print("⚠️ Secret Error: Make sure EMAIL_PASSWORD, EMAIL_ADDRESS, JWT_SECRET, and NGROK_AUTHTOKEN exist in Colab Secrets and have Notebook Access toggled ON.")

# 2. Configure Ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# 3. Clean up old processes so ports don't get blocked
ngrok.kill()
!pkill -f streamlit
time.sleep(2)

# 4. Start Streamlit in the background, explicitly passing the environment variables!
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=os.environ.copy(),  # 🚀 THIS IS THE MAGIC LINE THAT FIXES THE ERROR
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# 5. Open ngrok tunnel
try:
    public_url = ngrok.connect(8501).public_url
    print("=" * 60)
    print(f"🚀 Infosys Portal Live URL: {public_url}")
    print("=" * 60)
    print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

    while True:
        time.sleep(1)

except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")